In [ ]:
"""
Proyecto 4 NAND2TETRIS



Machine Language

Binary Code

Controla CPU, Registros y Memoria

Hack Computer Arquitecture

16-bit computer

CPU with ALU

Two Registers

A (Address Register) -> Hold memory and addresses

D (Data Register) -> Used for calculation

Memory Modules

ROM -> Program Instruction

RAM -> Read – Write Memory

Input / Output

Screen

Keyboard

Instructions

A – instruction (@value)

C – instruction (dest = comp; jump)

A – Instruction

Load 15-bit value

0 v v v v v v v v v v v v v v v  -> Binary representation 0 indicates A - Instruction

Load Constans into A- Instruction

Point memory addresses

Set up jumps for program flow

C – Instruction

Performs computations, memory access, and jumps.

Can compute values, store results, or control the flow of the program.

comp

Specifies the computation (like D+M, !D, A-1)

D+M

dest

Specifies where to store the result (A, D, M)

D=A

jump

Determines conditional/unconditional jumps

JGT

1 1 1 a c c c c c c d d d j j j  -> The first three bits (111) indicate a C-instruction

Memory Access  M Register  (Memory Register)

RAM has 32k memory (0 to 32767)

A- Instruction is used to access memory

The M register refers to RAM[A]

If A=100, then M means RAM[100]



Examples



//** Store 5 in RAM[10]



// Load address 10

@10

// Store value 5 at RAM[10]

M=5



//**  Copy Value from RAM[20] to RAM[30]



// Load address 20

@20

D=M   // D = RAM[20]

// Load address 30

@30

M=D   // RAM[30] = D



Control Flow (Jump Instructions) Solo 7 jumps

JGT (Jump if greater than 0)

JGE (Jump if greater than or equal to 0)

JLT (Jump if less than 0)

JLE (Jump if less than or equal to 0)

JEQ (Jump if equal to 0)

JNE (Jump if not equal to 0)

JMP (Unconditional jump)


"""

'\nProyecto 4 NAND2TETRIS\n\n\n\nMachine Language\n\nBinary Code\n\nControla CPU, Registros y Memoria\n\nHack Computer Arquitecture\n\n16-bit computer\n\nCPU with ALU\n\nTwo Registers\n\nA (Address Register) -> Hold memory and addresses\n\nD (Data Register) -> Used for calculation\n\nMemory Modules\n\nROM -> Program Instruction\n\nRAM -> Read – Write Memory\n\nInput / Output\n\nScreen\n\nKeyboard\n\nInstructions\n\nA – instruction (@value)\n\nC – instruction (dest = comp; jump)\n\nA – Instruction\n\nLoad 15-bit value\n\n0 v v v v v v v v v v v v v v v  -> Binary representation 0 indicates A - Instruction\n\nLoad Constans into A- Instruction\n\nPoint memory addresses\n\nSet up jumps for program flow\n\nC – Instruction\n\nPerforms computations, memory access, and jumps.\n\nCan compute values, store results, or control the flow of the program.\n\ncomp\n\nSpecifies the computation (like D+M, !D, A-1)\n\nD+M\n\ndest\n\nSpecifies where to store the result (A, D, M)\n\nD=A\n\njump\n\nDetermin

In [1]:
%%capture
from google.colab import drive
drive.mount("/content/drive/")

# Verificar que la carpeta existe
!ls "/content/drive/MyDrive/"

# Instalar import-ipynb
!pip install import-ipynb

# Cambiar a la carpeta correcta
%cd "/content/drive/MyDrive/Colab Notebooks/Trabajos U/"

# Importar archivos IPYNB
import import_ipynb
import Proyecto2  # Importar funciones desde Proyecto2.ipynb
import Proyecto3  # Importar funciones desde Proyecto3.ipynb

In [ ]:
import Proyecto2  # Importar funciones desde Proyecto2.ipynb
import Proyecto3  # Importar funciones desde Proyecto3.ipynb

In [ ]:
from Proyecto2 import ALU, Add16, And, Or, Not, Xor, FullAdder, Inc16
from Proyecto3 import Register, PC, RAM64K, RAM16K

In [ ]:
class CPU:
    def __init__(self):
        self.A = Register() #registro 16 bits
        self.D = Register() #registro 16 bits
        self.PC = PC()      # program counter
              # funcion ALU

    def execute(self, instruction, memory): # Funcion ejecutar, parametros instruccion y memoria, Instruccion registro de 16 bits, Memoria es RAM
        """Ejecuta una instrucción Hack."""
        print(f"Instrucción recibida: {instruction}")
        print(f"Primeros 3 bits de la instrucción: {instruction[:3]}")
        if instruction[0] == 0:  # A-instruction  # Si empieza con cero la instruccion 0 v v v ... se guarda el valor en registro A
            self.A.update(instruction, 1)
            print(f"Procesando A-instruction: {instruction}")
            self.PC.update([0]*16, 1, 0, 0)
            print(f"Salta ciclo de reloj a: {self.PC.update([0]*16, 0, 0, 0) }")

        if instruction[0] == 1 and instruction[1] == 1 and instruction[2] == 1:  # C-instruction de aritmetico logicas  # Si empieza con 1 la instruccion 1 v v v...
            print("Procesando C-instruction")
            comp = instruction[4:10]  # Bits de cómputo
            dest = instruction[10:13]  # Bits de destino
            jump = instruction[13:16]  # Bits de salto

            print(f"comp: {comp}, dest: {dest}, jump: {jump}")
            print(f"comp:{comp[0]},{comp[1]},{comp[2]},{comp[3]},{comp[4]},{comp[5]}")

            x = self.D.update([0]*16, 0)  # Guarda el Valor de D (Data Register)
            a_value = self.A.update([0]*16, 0)
            if(instruction[3] == 0):
              y = a_value  # y es igual a la instruccion A
              print(f"y toma el valor de A: {y}")
            if(instruction[3] == 1):
              y = memory.update([0]*16, 0, a_value)  # Guarda el valor de la memoria con direccion A, y = M[A]
              print(f"y toma el valor de la memoria con direccion A M[A]: {y}")

            # Ejecutar la ALU con los valores de entrada x (D) e y (memoria[A])
            result, zr, ng = ALU(x, y, comp[0],comp[1],comp[2],comp[3],comp[4],comp[5])   # se opera el registro D con el registro almancenado en memoria con direccion A

            print(f"ALU result: {result}, Zero flag: {zr}, Negative flag: {ng}")

            # Se verifica cual es el registro de destino

            if dest[0]:
              self.A.update(result, 1)  # dest[0] == 1 → Guardar en A
            if dest[1]:
              self.D.update(result, 1)  # dest[1] == 1 → Guardar en D
            if dest[2]:
              memory.update(result, 1, self.A.update([0]*16, 0)) # dest[2] == 1 → Guardar en la memoria en la dirección A

            if (jump[0] and ng) or (jump[1] and zr) or (jump[2] and not zr):  # jump[0] == 1 → Saltar si el resultado es negativo (ng == 1). # jump[1] == 1 → Saltar si el resultado es cero (zr == 1).  # jump[2] == 1 → Saltar si el resultado no es cero (zr == 0).
                print(f"Saltar a {self.A.update([0]*16, 0)}")
                self.PC.update(self.A.update([0]*16, 0), 0, 1, 0)             # Si se cumple alguna de estas condiciones, se actualiza el contador de programa (PC) con el valor de A:
            else:

                self.PC.update([0]*16, 1, 0, 0)                               # Si no se cumple la condición de salto, el PC simplemente se incrementa para ejecutar la siguiente instrucción secuencialmente:
                print(f"No se cumplio la condicion de salto, siguiente contador: {self.PC.update([0]*16, 0, 0, 0) }")


Ejemplo de ejecución en la CPU Hack

Supongamos que queremos ejecutar las siguientes instrucciones:

@5 → Cargar 5 en el registro A.

D=A → Copiar A en D.

@10 → Cargar 10 en A.

D=D+A → Sumar D + A y guardar en D.

@100 → Cargar 100 en A.

M=D → Guardar D en memoria[A] (posición 100).

In [ ]:
class HackComputer:
    def __init__(self):
        self.cpu = CPU()
        self.memory = RAM64K()
        self.rom = RAM64K()

    def load_program(self, program):
        """Carga un programa en la ROM."""  # Itera el programa a ejecutar
        for i, instruction in enumerate(program):
            address = [int(x) for x in format(i, '016b')]
            print(f"Cargando en ROM{[int(x) for x in format(i, '016b')]}: {instruction}")
            self.rom.update(instruction, 1, address)   #Lo guarda en la ROM


    def run(self):
        """Ejecuta el programa almacenado en ROM."""
        while True:
            address = self.cpu.PC.update([0]*16, 0, 0, 0) # Obtiene la dirección de la próxima instrucción desde PC (Program Counter).
            instruction = self.rom.update([0]*16, 0, address) # Obtiene la instrucción almacenada en rom[address].
            print(f"\n Ejecutando instrucción en ROM[{address}]: {instruction}")  # Depuración
            if instruction == [1]*16:  # Opcional: usar una convención para HALT
               print("Programa finalizado (HALT).")
               break
            self.cpu.execute(instruction, self.memory) # Ejecuta la instrucción con self.cpu.execute(instruction, self.memory).
            print(f"Después de ejecutar: A={self.cpu.A.update([0]*16, 0)}, D={self.cpu.D.update([0]*16, 0)}, Memoria con Address A={self.memory.update([0]*16, 0, self.cpu.A.update([0]*16, 0))}")






In [ ]:
# X es el registro D actulizado
# Y es el registro A o M[A]

program = [
    [0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,1],  # "@5" Empieza por cero guarda el valor en el registro A
    [1,1,1,0,1,1,0,0,0,0,0,1,0,0,0,0],  # D=A  Empieza por 1, Registro de computo ALU (X, Y , x = [0] * 16, x = [Not(bit) for bit in x],  y = [0] * 16, y = [Not(bit) for bit in y], ADD, AND , NEGAR SALIDA)
    [0,0,0,0,0,0,0,0,0,0,0,0,1,0,1,0],  # "@
    10"  Empieza por cero guarda el valor en el registro A
    [1,1,1,0,0,0,0,0,1,0,0,1,0,0,0,0],  # D=D+A
    [0,0,0,0,0,0,0,0,0,1,1,0,0,1,0,0],  # "A = @100"
    [1,1,1,0,0,0,1,1,0,0,0,0,1,0,0,0],  # M=D  A = 100 -> D = 15, para despues
    [1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1]   # HALT
]

In [ ]:
computer = HackComputer()
computer.load_program(program)
computer.run()

print("\nDespués de ejecutar el programa:")
print(f"A: {computer.cpu.A.update([0]*16, 0)} : ")
print(f"D: {computer.cpu.D.update([0]*16, 0)}")
print(f"Memoria con Address A: {computer.memory.update([0]*16, 0, computer.cpu.A.update([0]*16, 0))}")




Cargando en ROM[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]: [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 1]
Cargando en ROM[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1]: [1, 1, 1, 0, 1, 1, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0]
Cargando en ROM[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0]: [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 1, 0]
Cargando en ROM[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1]: [1, 1, 1, 0, 0, 0, 0, 0, 1, 0, 0, 1, 0, 0, 0, 0]
Cargando en ROM[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0]: [0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 0, 0, 1, 0, 0]
Cargando en ROM[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 1]: [1, 1, 1, 0, 0, 0, 1, 1, 0, 0, 0, 0, 1, 0, 0, 0]
Cargando en ROM[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 0]: [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]

 Ejecutando instrucción en ROM[[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]]: [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 1]
Instrucción recibida: [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 1]

In [ ]:
# Programa 2 para multiplicacion de dos numeros y desplegar inputs como el teclado usando ASCII



In [ ]:
class Memory:
    def __init__(self):
        self.ram = RAM16K()       # Memoria principal (0-16383)
        self.screen = Screen()    # Mapa de memoria de pantalla (16384-24575)
        self.keyboard = Keyboard() # Registro de teclado (24576)

    def update(self, data, load, address):
        """
        Actualiza/lee la memoria según la dirección.

        Args:
            data: Valor de 16 bits a escribir (lista)
            load: 1 para escribir, 0 para leer
            address: Dirección (entero o lista de bits)

        Returns:
            Valor leído de la memoria (lista de 16 bits)
        """
        # Convertir address a entero si es una lista de bits
        if isinstance(address, list):
            address = self._bits_to_int(address)

        # Determinar qué componente acceder
        if address < 16384:  # RAM16K
            return self._handle_ram(data, load, address)
        elif address < 24576:  # Screen
            return self._handle_screen(data, load, address - 16384)
        elif address == 24576:  # Keyboard
            return self._handle_keyboard(data, load)
        else:
            print(f"Error: Dirección inválida {address}")
            return [0]*16

    def _bits_to_int(self, bits):
        """Convierte lista de bits a entero"""
        return int(''.join(map(str, bits)), 2)

    def _handle_ram(self, data, load, address):
        """Maneja acceso a RAM16K"""
        # RAM16K espera dirección como lista de 14 bits
        if isinstance(address, int):
            address = [int(x) for x in f"{address:014b}"]
        return self.ram.update(data, load, address)

    def _handle_screen(self, data, load, address):
        """Maneja acceso a Screen"""
        # Screen usa dirección entera
        if isinstance(address, list):
            address = self._bits_to_int(address)
        return self.screen.update(data, load, address)

    def _handle_keyboard(self, data, load):
        """Maneja acceso a Keyboard"""
        if load:
            print("Warning: Keyboard es de solo lectura")
        return self.keyboard.update([0]*16, 0)

In [ ]:
class Screen:
    def __init__(self):
        # 8K words = 8192 registros de 16 bits
        self.memory = [[0]*16 for _ in range(8192)]

    def update(self, data, load, address):
        """Actualiza/lee la memoria de pantalla"""
        # Asegurarse de que address es entero
        if isinstance(address, list):
            address = int(''.join(map(str, address)), 2)

        if address < 0 or address >= 8192:
            print(f"Error: Dirección de pantalla inválida {address}")
            return [0]*16

        if load:
            self.memory[address] = data.copy()
        return self.memory[address].copy()

In [ ]:
class Keyboard:
    def __init__(self):
        self.key_code = [0]*16  # Almacena código ASCII de la última tecla

    def update(self, data, load):
        """Solo lectura - devuelve el código de tecla actual"""
        return self.key_code.copy()

    def set_key(self, key_code):
        """Simula presión de tecla (para pruebas)"""
        self.key_code = [int(x) for x in format(key_code, '016b')]

In [ ]:
# Ejemplo de uso desde el CPU
memory = Memory()

# Escribir en RAM[100]
memory.update([0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,1], 1, 100)  # Escribe 5 en RAM[100]

# Leer de Screen (posición 0 en memoria de pantalla = dirección 16384)
screen_pixel = memory.update([0]*16, 0, 16384)

# Leer teclado
key = memory.update([0]*16, 0, 24576)

class TestMemory:
    @staticmethod
    def test_memory():
        print("\nProbando Memory...")
        mem = Memory()

        # Test RAM
        print("Probando RAM16K...")
        test_data = [0,1,0,1,0,1,0,1,0,1,0,1,0,1,0,1]
        mem.update(test_data, 1, 100)  # Escribir en RAM[100]
        result = mem.update([0]*16, 0, 100)  # Leer de RAM[100]
        assert result == test_data, f"Error en RAM16K. Esperado: {test_data}, Obtenido: {result}"

        # Test Screen
        print("Probando Screen...")
        screen_data = [1,0,1,0,1,0,1,0,1,0,1,0,1,0,1,0]
        mem.update(screen_data, 1, 16384)  # Escribir en Screen[0]
        result = mem.update([0]*16, 0, 16384)  # Leer de Screen[0]
        assert result == screen_data, f"Error en Screen. Esperado: {screen_data}, Obtenido: {result}"

        # Test Keyboard
        print("Probando Keyboard...")
        mem.keyboard.set_key(65)  # Simular tecla 'A' (ASCII 65)
        result = mem.update([0]*16, 0, 24576)
        expected = [0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,1]  # 65 en binario
        assert result == expected, f"Error en Keyboard. Esperado: {expected}, Obtenido: {result}"

        print("✔ Todas las pruebas de Memory pasaron correctamente!")

# Ejecutar pruebas
TestMemory.test_memory()


Probando Memory...
Probando RAM16K...
Probando Screen...
Probando Keyboard...
✔ Todas las pruebas de Memory pasaron correctamente!
